In [2]:
# ==== Wav2Vec2 ZERO-SHOT EVALUATION (no fine-tuning) ====
# Baseline: Pretrained Wav2Vec2 with a randomly initialized 2-class head
# Outputs: accuracy, macro-F1, AUROC, classification report, confusion matrix, per-file predictions

import os
import glob
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset
from transformers import AutoProcessor, Wav2Vec2ForSequenceClassification, Trainer, TrainingArguments
import evaluate
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# --------------------
# Reproducibility
# --------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --------------------
# Paths
# --------------------
CHECKPOINT   = "facebook/wav2vec2-base"       # you can swap this for other w2v2 checkpoints
OUTPUT_DIR   = "./wav2vec2-zero-shot"         # <-- zero-shot outputs will go here
# TRAIN_DIR    = r"D:\Thesis\Song\dataset\Train"
# VALID_DIR    = r"D:\Thesis\Song\dataset\Valid"
TEST_DIR     = r"D:\Thesis\Song\Speech\test"

# --------------------
# Dataset definition
# --------------------
class Wav2VecDataset(Dataset):
    def __init__(self, root_dir, processor, target_sr=16000, max_duration=10.0, recursive=True):
        self.files = []
        self.labels = []
        self.processor = processor
        self.target_sr = target_sr
        self.max_length = int(target_sr * max_duration)  # e.g. 160000 samples (10 s)

        # Find .wav files (recursive if requested)
        if recursive:
            candidates = glob.glob(os.path.join(root_dir, "**", "*.wav"), recursive=True)
            candidates += glob.glob(os.path.join(root_dir, "**", "*.WAV"), recursive=True)
        else:
            candidates = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                          if f.lower().endswith(".wav")]

        for path in sorted(candidates):
            parent = os.path.basename(os.path.dirname(path)).lower()
            if parent == "real":
                label = 0
            elif parent == "fake":
                label = 1
            else:
                continue  # ignore files not inside Real/ or Fake folders

            self.files.append(path)
            self.labels.append(label)

        print(f"✅ Loaded {len(self.files)} files from {root_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        label = self.labels[idx]

        waveform, sr = torchaudio.load(path)

        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)

        waveform = waveform.squeeze().numpy()

        # Pad or truncate to fixed length (10 s)
        if len(waveform) > self.max_length:
            waveform = waveform[:self.max_length]
        else:
            pad = self.max_length - len(waveform)
            waveform = np.pad(waveform, (0, pad), mode="constant")

        # Processor packs into input_values
        inputs = self.processor(
            raw_speech=waveform,
            sampling_rate=self.target_sr,
            return_tensors="pt",
            padding=False
        )

        return {
            "input_values": inputs["input_values"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


# --------------------
# Load processor & model
# --------------------
processor = AutoProcessor.from_pretrained(CHECKPOINT)

# Load pretrained model with a fresh 2-class head (random init)
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2
)
# Optional: freeze the base to emphasize "zero-shot-ness" (head stays random anyway)
# for p in model.wav2vec2.parameters():
#     p.requires_grad = False

# --------------------
# Build datasets (we only need TEST for zero-shot eval, but building all is fine)
# --------------------
test_dataset = Wav2VecDataset(TEST_DIR, processor, target_sr=16000, max_duration=10.0, recursive=True)

# --------------------
# Metrics
# --------------------
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
roc_metric = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    # Prob for class 1 (Fake)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "auroc": roc_metric.compute(prediction_scores=probs, references=labels)["roc_auc"]
    }

# --------------------
# Trainer (evaluate only)
# --------------------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_eval_batch_size=32,
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    eval_dataset=test_dataset,
    tokenizer=processor,  # fine for now; HF will deprecate in favor of processing_class later
    compute_metrics=compute_metrics
)

# --------------------
# Run evaluation (aggregate metrics)
# --------------------
print("Evaluating zero-shot Wav2Vec2 on test set...")
metrics = trainer.evaluate(test_dataset)

print("\n=== Zero-Shot Test Metrics (Wav2Vec2) ===")
for k, v in metrics.items():
    if k.startswith("eval_"):
        print(f"{k[5:]}: {v:.4f}")

# --------------------
# Detailed report: predictions, classification report, confusion matrix
# --------------------
predictions = trainer.predict(test_dataset)
logits = predictions.predictions
labels = predictions.label_ids
y_pred = np.argmax(logits, axis=1)
probs_fake = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

# Classification report
target_names = ["Real (0)", "Fake (1)"]
rep_str = classification_report(labels, y_pred, target_names=target_names, digits=4)
print("\n=== Classification Report (Zero-Shot Wav2Vec2) ===")
print(rep_str)

# Confusion matrix
cm = confusion_matrix(labels, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm,
    index=["True Real (0)", "True Fake (1)"],
    columns=["Pred Real (0)", "Pred Fake (1)"]
)
print("\n=== Confusion Matrix (Zero-Shot Wav2Vec2) ===")
print(cm_df)

# --------------------
# Save detailed artifacts
# --------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1) metrics.txt
with open(os.path.join(OUTPUT_DIR, "metrics_zero_shot.txt"), "w", encoding="utf-8") as f:
    f.write("==== Zero-Shot Test Metrics (Wav2Vec2) ====\n")
    for k, v in metrics.items():
        f.write(f"{k}: {v}\n")

# 2) classification report (txt + csv)
with open(os.path.join(OUTPUT_DIR, "classification_report_test.txt"), "w", encoding="utf-8") as f:
    f.write(rep_str)

rep_df = pd.DataFrame(classification_report(labels, y_pred, target_names=target_names, output_dict=True)).transpose()
rep_df.to_csv(os.path.join(OUTPUT_DIR, "classification_report_test.csv"), index=True)

# 3) confusion matrix
cm_df.to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_test.csv"), index=True)

# 4) per-file predictions
per_file_df = pd.DataFrame({
    "Filename": [os.path.basename(p) for p in test_dataset.files],
    "TrueLabel": ["Real" if t == 0 else "Fake" for t in labels],
    "PredLabel": ["Real" if p == 0 else "Fake" for p in y_pred],
    "Prob_Fake": probs_fake
})
per_file_df.to_csv(os.path.join(OUTPUT_DIR, "per_file_predictions.csv"), index=False)

print(f"\n✅ Saved zero-shot artifacts to: {os.path.abspath(OUTPUT_DIR)}")


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Loaded 2544 files from D:\Thesis\Song\Speech\test


C:\Users\j3n50\AppData\Local\Temp\ipykernel_13356\1919362806.py:154: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluating zero-shot Wav2Vec2 on test set...


c:\Users\j3n50\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:98: UserWarning: Using `raw_speech` as a keyword argument is deprecated. Use `audio` instead.
  warnings.warn("Using `raw_speech` as a keyword argument is deprecated. Use `audio` instead.")



=== Zero-Shot Test Metrics (Wav2Vec2) ===
loss: 0.6908
model_preparation_time: 0.0020
accuracy: 0.5605
f1_macro: 0.5393
auroc: 0.6623
runtime: 39.9848
samples_per_second: 63.6240
steps_per_second: 2.0010


c:\Users\j3n50\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:98: UserWarning: Using `raw_speech` as a keyword argument is deprecated. Use `audio` instead.
  warnings.warn("Using `raw_speech` as a keyword argument is deprecated. Use `audio` instead.")



=== Classification Report (Zero-Shot Wav2Vec2) ===
              precision    recall  f1-score   support

    Real (0)     0.5515    0.7573    0.6382      1302
    Fake (1)     0.5820    0.3543    0.4404      1242

    accuracy                         0.5605      2544
   macro avg     0.5667    0.5558    0.5393      2544
weighted avg     0.5664    0.5605    0.5416      2544


=== Confusion Matrix (Zero-Shot Wav2Vec2) ===
               Pred Real (0)  Pred Fake (1)
True Real (0)            986            316
True Fake (1)            802            440

✅ Saved zero-shot artifacts to: d:\Thesis\Song\ZershotSPeech\wav2vec2-zero-shot
